# Grow a regression tree in NumPy

`DecisionTreeRegressor` searches every feature and every threshold for the split that
leaves the least squared error, then does the same in each child. Here you write that
search and the recursion around it, with the two stopping rules `max_depth` and
`min_samples_leaf`, and check your tree against small cases where the usual shortcuts
go wrong.

**How to work through it.** Part 1 asks for one function and Part 2 asks for two. Replace
each `TODO` and `raise NotImplementedError` with your code, then run the check cell under
it. The check prints `Part N passed` when your function is right, and stops with a message
that says what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

def sse(v):
    """Sum of squared deviations from the mean (0 for an empty array)."""
    return float(((v - v.mean()) ** 2).sum()) if len(v) else 0.0

rng = np.random.default_rng(11)
# 80 training rows and 40 new rows, every feature between 0 and 10.
X = rng.uniform(0, 10, (80, 3))      # the third feature is unrelated to the target
# A wavy effect of the first feature, a slope in the second, and noise.
y = 3 * np.sin(X[:, 0]) + 0.5 * X[:, 1] + rng.normal(0, 0.5, 80)
X_new = rng.uniform(0, 10, (40, 3))
print("training rows:", X.shape, " new rows:", X_new.shape)

## Part 1 (coding task): The best split

Write `best_split(X, y, min_leaf=1)`. For every feature $j$ and every threshold $t$
halfway between two neighboring **distinct** values of that feature, the split sends
the rows with $x_j \le t$ left and the rest right. Among the splits that leave at least
`min_leaf` rows on each side, return `(j, t, cost)` for the one with the smallest

$$
\text{cost} = \text{sse}(\text{left targets}) + \text{sse}(\text{right targets}),
$$

where `sse` is the helper from the setup. On equal cost keep the first one found,
going through the features in order and the thresholds from small to large. Return
`None` when no split is allowed.

In [ ]:
def best_split(X, y, min_leaf=1):
    """(feature, threshold, cost) of the split with the least total squared error, or None."""
    # TODO
    raise NotImplementedError

In [ ]:
x8 = np.arange(1.0, 9.0)[:, None]
s = best_split(x8, np.array([1.0, 2.0, 1.0, 2.0, 7.0, 8.0, 7.0, 0.0]))
assert s is not None and len(s) == 3, "return a tuple (feature, threshold, cost)"
assert not np.any(np.isclose(s[1], x8)), (f"threshold {s[1]} is one of the data values: "
                                          "put each threshold halfway between two neighboring distinct values")
assert np.isclose(s[1], 4.5), (f"threshold {s[1]}, expected 4.5: compare the total squared error of the children, "
                               "not their variances, which treat one row and seven rows as equals")
assert np.isclose(s[2], 42.0), f"cost {s[2]}, expected 42.0"
dup = best_split(np.array([[1.0], [2.0], [2.0], [2.0], [3.0], [4.0]]), np.array([0.0, 0.0, 0.0, 6.0, 6.0, 6.0]))
assert not np.isclose(dup[1], 2.0), ("threshold 2.0 cannot separate the three rows at 2.0: split only between "
                                     "distinct values, so equal values always go to the same side")
assert np.isclose(dup[1], 2.5) and np.isclose(dup[2], 27.0), f"expected (0, 2.5, 27.0), got {dup}"
y_out = np.array([2.0, 1.0, 2.0, 1.0, 6.0, 7.0, 6.0, 15.0])
assert np.isclose(best_split(x8, y_out)[1], 7.5), "with min_leaf=1 the best split isolates the last row"
assert np.isclose(best_split(x8, y_out, min_leaf=3)[1], 4.5), "with min_leaf=3 each side needs at least 3 rows"
assert best_split(x8[:4], np.zeros(4), min_leaf=3) is None, "no split leaves 3 rows on both sides of 4 rows: return None"
two = best_split(np.c_[np.zeros(6), np.arange(6.0)], np.array([0.0, 0, 0, 5, 5, 5]))
assert two[0] == 1 and np.isclose(two[1], 2.5), f"a constant feature offers no split, the second one does: got {two}"
tie = best_split(np.array([[0.0, 0.0], [1.0, 1.0], [2.0, 2.0], [3.0, 3.0]]), np.array([0.0, 0.0, 0.0, 1.0]))
assert tie[0] == 0 and np.isclose(tie[1], 2.5) and np.isclose(tie[2], 0.0), (
    f"got {tie}: both features separate the labels perfectly at threshold 2.5 with cost 0, so the "
    "tie rule decides between them. Keep the first one found, feature 0, not the last")
print("Part 1 passed")

## Part 2 (coding task): Grow the tree and predict

Write `grow(X, y, max_depth, min_leaf=1)`, returning the tree as nested dictionaries: a
leaf is `{"leaf": mean of its targets}` and an internal node is
`{"feature": j, "threshold": t, "left": ..., "right": ...}`, with $x_j \le t$ going left.
A node becomes a leaf when it is at depth `max_depth` (the root is at depth 0), when its
targets are all equal, or when `best_split` finds no allowed split. Then write
`predict(tree, X)`, returning one prediction per row.

In [ ]:
def grow(X, y, max_depth, min_leaf=1, depth=0):
    """Nested-dictionary regression tree."""
    # TODO
    raise NotImplementedError

def predict(tree, X):
    """One prediction per row of X."""
    # TODO
    raise NotImplementedError

In [ ]:
def n_leaves(tree):
    return 1 if "leaf" in tree else n_leaves(tree["left"]) + n_leaves(tree["right"])

def depth_of(tree):
    return 0 if "leaf" in tree else 1 + max(depth_of(tree["left"]), depth_of(tree["right"]))

stump = grow(np.arange(1.0, 7.0)[:, None], np.array([1.0, 4.0, 7.0, 8.0, 11.0, 17.0]), max_depth=1)
assert "leaf" not in stump, "a depth-1 tree on varied targets should split once"
assert depth_of(stump) == 1, f"max_depth=1 should split once, at the root (depth 0); this tree is {depth_of(stump)} splits deep"
assert np.allclose(predict(stump, np.array([[2.0], [4.4], [4.6], [100.0]])), [5, 5, 14, 14]), \
    "the leaves should predict the mean of their training targets"
flat = grow(X[:5], np.full(5, 3.0), max_depth=4)
assert flat == {"leaf": 3.0}, "a node whose targets are all equal should be a leaf"
tree3 = grow(X, y, max_depth=3)
assert depth_of(tree3) == 3, f"max_depth=3 should give a tree 3 splits deep, got {depth_of(tree3)}"
assert n_leaves(tree3) == 8, f"expected 8 leaves at max_depth=3, got {n_leaves(tree3)}"
expected3 = [3.10004, 0.12501, 3.10004, 1.55972, 3.10004, 0.12501, 5.69837, 0.12501]
assert np.allclose(predict(tree3, X_new[:8]), expected3, atol=1e-3), \
    f"predictions on new rows {np.round(predict(tree3, X_new[:8]), 4).tolist()}, expected {expected3}"
tree_leaf = grow(X, y, max_depth=4, min_leaf=5)
assert n_leaves(tree_leaf) == 12, f"expected 12 leaves at max_depth=4, min_leaf=5, got {n_leaves(tree_leaf)}"
train_err = np.mean((predict(tree_leaf, X) - y) ** 2)
assert np.isclose(train_err, 1.24772, atol=1e-4), f"training MSE {train_err:.4f}, expected 1.2477: check how min_leaf limits the splits"
print("Part 2 passed")

## Solution

### Part 1: The best split

In [ ]:
def best_split(X, y, min_leaf=1):
    """(feature, threshold, cost) of the split with the least total squared error, or None."""
    best = None
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])                      # sorted, duplicates removed
        for t in (values[:-1] + values[1:]) / 2:
            left = X[:, j] <= t
            if left.sum() < min_leaf or (~left).sum() < min_leaf:
                continue
            cost = sse(y[left]) + sse(y[~left])
            if best is None or cost < best[2] - 1e-12:
                best = (j, float(t), cost)
    return best

**Why it works.** `np.unique` sorts the values and drops repeats, so every threshold lies strictly
between two different values and the mask `X[:, j] <= t` is exactly the partition whose
cost is computed. Looping over sorted positions instead would also try to cut between
the three rows at 2.0, a split no threshold can make. The cost adds the children's
squared errors, which weights each child by its size. The variances would rate a child
of one row, with variance 0, as good as a pure child of seven. The `- 1e-12` keeps the
first of two splits whose costs differ only by rounding.

### Part 2: Grow the tree and predict

In [ ]:
def grow(X, y, max_depth, min_leaf=1, depth=0):
    """Nested-dictionary regression tree."""
    split = None
    if depth < max_depth and sse(y) > 0:
        split = best_split(X, y, min_leaf)
    if split is None:
        return {"leaf": float(y.mean())}
    j, t, _ = split
    left = X[:, j] <= t
    return {"feature": j, "threshold": t,
            "left": grow(X[left], y[left], max_depth, min_leaf, depth + 1),
            "right": grow(X[~left], y[~left], max_depth, min_leaf, depth + 1)}

def predict(tree, X):
    """One prediction per row of X."""
    out = np.empty(len(X))
    for i, row in enumerate(X):
        node = tree
        while "leaf" not in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        out[i] = node["leaf"]
    return out

**Why it works.** The recursion carries the depth down the tree, and a node stops when it reaches
`max_depth`, when it is already pure, or when `best_split` returns `None`, which is how
`min_leaf` reaches the leaves: no split that would leave a child under `min_leaf` rows is
ever offered. A prediction walks from the root with the same test, $x_j \le t$, that
built the split, so a training row always lands in a leaf that contains it. The leaf
predicts a mean of training targets, so the tree can never predict outside their range,
and these trees agree with `DecisionTreeRegressor(max_depth=..., min_samples_leaf=...)`
from scikit-learn on every one of the 40 new rows.